# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


## Крок 0. Завантаження даних

Є три способи дістати дані — оберіть будь-який.

**Спосіб A — Kaggle API (рекомендований).** Завантаження з Kaggle API. Зручно, бо декілька файлів і вони завантажаться всі самостійно. Для цього способу завантажте свій `kaggle.json` (Kaggle → Account → Create New API Token), потім виконайте:
```python
from google.colab import files; files.upload()   # оберіть kaggle.json
```
і розкоментуйте відповідний блок нижче.

**Спосіб B — ручне завантаження.** Завантажте архів з посилання на датасет вище з Kaggle, розпакуйте і покладіть `ratings.csv`, `books.csv`, `book_tags.csv`, `tags.csv` поруч із ноутбуком (або через панель Files у Colab).

**Спосіб C — GitHub-дзеркало (фолбек).** Оригінальний автор виклав файли і на GitHub — код нижче підхопить їх автоматично, якщо локально файлів немає.


In [ ]:
from google.colab import files; files.upload()

In [2]:
# (Спосіб A) Kaggle API — розкоментуйте, якщо завантажили kaggle.json
!pip -q install kaggle
import os, shutil
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
!kaggle datasets download -d zygmunt/goodbooks-10k --unzip -p .

Dataset URL: https://www.kaggle.com/datasets/zygmunt/goodbooks-10k
License(s): CC-BY-SA-4.0
100% 11.6M/11.6M [00:01<00:00, 7.05MB/s]



In [3]:
import os
import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]

def load(fname):
    """Спочатку шукаємо файл локально, інакше тягнемо з GitHub-дзеркала."""
    if os.path.exists(fname):
        return pd.read_csv(fname)
    print(f"{fname} не знайдено локально — завантажую з GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")

ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings: (981756, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,2767052,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,41865,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,2657,Harper Lee,To Kill a Mockingbird,4.25
4,4671,F. Scott Fitzgerald,The Great Gatsby,3.89


## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [4]:
books["goodreads_book_id"] = books["best_book_id"]

In [5]:
books.head()

,id,book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,goodreads_book_id
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,2767052
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,3
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,41865
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,2657
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,4671


In [6]:
# Канонічні жанри, які шукаємо серед тегів
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 9715 / 10000

Розподіл жанрів:
genre
contemporary       5128
fantasy            4179
romance            4119
mystery            3601
young-adult        3555
classics           2737
historical         2484
thriller           2449
science-fiction    2170
crime              2027
nonfiction         1800
horror             1328
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1.0,1,1,1,0,0,0,0,1,0,1,1,0
2.0,1,1,1,0,0,0,0,1,0,0,0,0
3.0,1,0,1,0,0,0,0,1,0,1,1,0
5.0,1,0,1,0,0,0,0,1,0,1,1,0
6.0,1,0,1,0,0,0,0,1,0,1,1,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [12]:
TOP_BOOKS = 4000       # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 6000         # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4     # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 11,005 | користувачів: 3,235 | книг: 115
Щільність: 0.0296


In [13]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 5,787 | користувачів з val-лайками: 1,014


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [14]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.

In [15]:
import torch
import torch.nn.functional as F

In [16]:
item_feats_tensor = torch.tensor(genre_matrix.values, dtype=torch.float32)
item_emb = F.normalize(item_feats_tensor, p=2, dim=1)

In [17]:
def user_vector(user_idx):
    real_user_id = users[user_idx]
    user_likes = train_df[(train_df["user_id"] == real_user_id) & (train_df["rating"] >= LIKE_THRESHOLD)]

    if len(user_likes) == 0:
        return torch.zeros(n_genres)

    liked_item_idxs = [item_to_idx[b] for b in user_likes["book_id"]]
    weights = user_likes["rating"].values
    item_embeddings_np = item_emb.numpy()
    user_item_embs = [item_embeddings_np[item_id] for item_id in liked_item_idxs]
    user_emb_np = np.average(
        user_item_embs,
        weights= weights,
        axis=0)

    u_vec = torch.tensor(user_emb_np, dtype=torch.float32)
    return F.normalize(u_vec, p=2, dim=0)

In [18]:
def vsm_scores(user_idxs):
    user_vectors_list = []
    for u in user_idxs.tolist():
        user_vectors_list.append(user_vector(u))
    u_matrix = torch.stack(user_vectors_list)
    scores = torch.matmul(u_matrix, item_emb.t())
    return scores


In [19]:
vsm_recall = recall_at_k(vsm_scores, k=10)
vsm_recall

0.09023066485753053

In [22]:
test_user_idx = list(val_pos.keys())[2]
real_demo_uid = users[test_user_idx]

with torch.no_grad():
    user_score = vsm_scores(torch.tensor([test_user_idx]))[0].clone()

for idx in seen_by_user[test_user_idx]:
    user_score[idx] = -1e9

top5_indices = torch.topk(user_score, k=5).indices.tolist()

for rank, idx in enumerate(top5_indices, start=1):
    book_id = items[idx]
    title = title_of.get(book_id)
    print(f"{rank}. {title} ")

1. Men Are from Mars, Women Are from Venus 
2. Warrior of the Light 
3. The Virtue of Selfishness: A New Concept of Egoism 
4. Slouching Towards Bethlehem 
5. Sailing Alone Around the Room: New and Selected Poems 


**Питання:** Recall@10 у векторного підходу досить низький. Чому?


На низке значення Recall@10  може вплинути використання тільки параметру жанру книг, без інших параметрів.

---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


In [23]:
torch.manual_seed(42)
np.random.seed(42)
import torch.nn as nn

In [24]:
class TwoTowerModel(nn.Module):
    def __init__(self, n_users, n_genres, item_features, embedding_dim=16):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_projection = nn.Linear(n_genres, embedding_dim)
        self.item_features = nn.Parameter(item_features, requires_grad=False)

        self.temperature = 10.0

        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_projection.weight, std=0.1)

    def get_all_item_embeddings(self):
        raw_item_embs = self.item_projection(self.item_features)
        return F.normalize(raw_item_embs, p=2, dim=1)

    def forward(self, user_idx):
        u_emb = self.user_embedding(user_idx)
        u_emb_norm = F.normalize(u_emb, p=2, dim=1)
        i_emb_norm = self.get_all_item_embeddings()
        scores = torch.matmul(u_emb_norm, i_emb_norm.t()) * self.temperature
        return scores

In [25]:
embedding_dim = 16
tt_model = TwoTowerModel(len(users), n_genres, item_feats, embedding_dim)

In [26]:
optimizer = torch.optim.Adam(tt_model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()
epochs = 100
batch_size = 256
n_pos = len(pos_u)

for epoch in range(1, epochs + 1):
    tt_model.train()

    neg_i_list = []
    for u in pos_u.tolist():
        while True:
            neg_item = np.random.randint(0, M)
            if neg_item not in seen_by_user[u]:
                neg_i_list.append(neg_item)
                break
    neg_i = torch.tensor(neg_i_list)

    indices = torch.randperm(n_pos)
    epoch_loss = 0

    for start_idx in range(0, n_pos, batch_size):
        batch_idx = indices[start_idx : start_idx + batch_size]

        b_users = pos_u[batch_idx]
        b_pos_items = pos_i[batch_idx]
        b_neg_items = neg_i[batch_idx]

        u_emb = F.normalize(tt_model.user_embedding(b_users), p=2, dim=1)
        all_items_emb = tt_model.get_all_item_embeddings()

        pos_i_emb = all_items_emb[b_pos_items]
        neg_i_emb = all_items_emb[b_neg_items]


        pos_scores = (u_emb * pos_i_emb).sum(dim=1) * tt_model.temperature
        neg_scores = (u_emb * neg_i_emb).sum(dim=1) * tt_model.temperature


        predictions = torch.cat([pos_scores, neg_scores])
        targets = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])

        loss = criterion(predictions, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(batch_idx)

    if epoch % 20 == 0 or epoch == 1:
        tt_model.eval()

        score_fn = lambda u_tensor: tt_model(u_tensor)
        recall = recall_at_k(score_fn, k=10)
        print(f"Епоха {epoch:03d} | Середній Loss: {epoch_loss/n_pos:.4f} | Recall@10: {recall:.4f}")

Епоха 001 | Середній Loss: 1.0804 | Recall@10: 0.0855
Епоха 020 | Середній Loss: 0.3803 | Recall@10: 0.0970
Епоха 040 | Середній Loss: 0.3379 | Recall@10: 0.0875
Епоха 060 | Середній Loss: 0.3277 | Recall@10: 0.0896
Епоха 080 | Середній Loss: 0.3226 | Recall@10: 0.0909
Епоха 100 | Середній Loss: 0.3132 | Recall@10: 0.0889


In [28]:
tt_model.eval()
tt_recall = recall_at_k(lambda u_tensor: tt_model(u_tensor), k=10)
tt_recall

test_user_idx = list(val_pos.keys())[2]
real_demo_uid = users[test_user_idx]

print(f"Топ-5 рекомендацій для користувача №{real_demo_uid}")
with torch.no_grad():
    user_score = tt_model(torch.tensor([test_user_idx]))[0].clone()

for idx in seen_by_user[test_user_idx]:
    user_score[idx] = -1e9

top5_indices = torch.topk(user_score, k=5).indices.tolist()

for rank, idx in enumerate(top5_indices, start=1):
    book_id = items[idx]
    title = title_of.get(book_id)
    print(f"{rank}. {title} ")

Топ-5 рекомендацій для користувача №25988
1. Digging to America 
2. Moon Palace 
3. Timbuktu 
4. Play It as It Lays 
5. The Broken Wings 


---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


In [29]:
class NCFModel(nn.Module):
    def __init__(self, n_users, n_genres, item_features, embedding_dim=16):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_features = nn.Parameter(item_features, requires_grad=False)
        input_dim = embedding_dim + n_genres
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1))

        nn.init.normal_(self.user_embedding.weight, std=0.1)

    def forward(self, user_idx):

        batch_size = user_idx.shape[0]
        M = self.item_features.shape[0]
        u_emb = self.user_embedding(user_idx)

        u_emb_expanded = u_emb.unsqueeze(1).expand(-1, M, -1)
        i_feats_expanded = self.item_features.unsqueeze(0).expand(batch_size, -1, -1)
        concat_vec = torch.cat([u_emb_expanded, i_feats_expanded], dim=2)

        scores = self.mlp(concat_vec).squeeze(2)
        return scores

In [30]:
ncf_model = NCFModel(len(users), n_genres, item_feats, embedding_dim=16)

In [31]:
optimizer = torch.optim.Adam(ncf_model.parameters(), lr=0.005)
criterion = nn.BCEWithLogitsLoss()
epochs = 100
batch_size = 256
n_pos = len(pos_u)

for epoch in range(1, epochs + 1):
    ncf_model.train()


    neg_i_list = []
    for u in pos_u.tolist():
        while True:
            neg_item = np.random.randint(0, M)
            if neg_item not in seen_by_user[u]:
                neg_i_list.append(neg_item)
                break
    neg_i = torch.tensor(neg_i_list)

    indices = torch.randperm(n_pos)
    epoch_loss = 0

    for start_idx in range(0, n_pos, batch_size):
        batch_idx = indices[start_idx : start_idx + batch_size]

        b_users = pos_u[batch_idx]
        b_pos_items = pos_i[batch_idx]
        b_neg_items = neg_i[batch_idx]


        u_emb = ncf_model.user_embedding(b_users)
        pos_i_feats = ncf_model.item_features[b_pos_items]
        neg_i_feats = ncf_model.item_features[b_neg_items]

        pos_concat = torch.cat([u_emb, pos_i_feats], dim=1)
        neg_concat = torch.cat([u_emb, neg_i_feats], dim=1)

        pos_scores = ncf_model.mlp(pos_concat).squeeze(1)
        neg_scores = ncf_model.mlp(neg_concat).squeeze(1)

        predictions = torch.cat([pos_scores, neg_scores])
        targets = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])

        loss = criterion(predictions, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(batch_idx)

    if epoch % 15 == 0 or epoch == 1:
        ncf_model.eval()
        score_fn = lambda u_tensor: ncf_model(u_tensor)
        recall = recall_at_k(score_fn, k=10)
        print(f"Епоха {epoch:02d} | Середній Loss: {epoch_loss/n_pos:.4f} | Recall@10: {recall:.4f}")



Епоха 01 | Середній Loss: 0.6933 | Recall@10: 0.0821
Епоха 15 | Середній Loss: 0.4778 | Recall@10: 0.1160
Епоха 30 | Середній Loss: 0.3541 | Recall@10: 0.1153
Епоха 45 | Середній Loss: 0.2938 | Recall@10: 0.1180
Епоха 60 | Середній Loss: 0.2794 | Recall@10: 0.1079
Епоха 75 | Середній Loss: 0.2559 | Recall@10: 0.1099
Епоха 90 | Середній Loss: 0.2483 | Recall@10: 0.1106


In [32]:
def rank_ncf(user_idx, candidate_idxs):
    ncf_model.eval()
    with torch.no_grad():

        u_emb = ncf_model.user_embedding(torch.tensor([user_idx])).expand(len(candidate_idxs), -1)

        i_feats = ncf_model.item_features[candidate_idxs]

        concat_vec = torch.cat([u_emb, i_feats], dim=1)
        logits = ncf_model.mlp(concat_vec).squeeze(1)
        probs = torch.sigmoid(logits).tolist()

    ranked_candidates = sorted(zip(candidate_idxs, probs), key=lambda x: x[1], reverse=True)
    return ranked_candidates



In [33]:
demo_user_idx = list(val_pos.keys())[2]
real_demo_uid = users[demo_user_idx]

np.random.seed(10)
random_candidates = np.random.choice(range(M), size=10, replace=False).tolist()

print(f"Перелік для користувача №{real_demo_uid}")
for idx in random_candidates:
    print(f" - {title_of.get(items[idx])}")

# Запускаємо ранжування
ranked_list = rank_ncf(demo_user_idx, random_candidates)

print("\nВідранжований список:")
for rank, (idx, prob) in enumerate(ranked_list, start=1):
    title = title_of.get(items[idx])
    print(f"{rank}. {title}")

Перелік для користувача №25988
 - The Broker
 - Warrior of the Light
 - No Logo
 - The Odyssey
 - The Egypt Game (Game, #1)
 - Giada's Family Dinners
 - War and Peace
 - The Iliad/The Odyssey
 - Moon Palace
 - I'm a Stranger Here Myself: Notes on Returning to America after Twenty Years Away

Відранжований список:
1. The Odyssey
2. Giada's Family Dinners
3. No Logo
4. I'm a Stranger Here Myself: Notes on Returning to America after Twenty Years Away
5. The Iliad/The Odyssey
6. War and Peace
7. Moon Palace
8. The Broker
9. Warrior of the Light
10. The Egypt Game (Game, #1)


---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


In [34]:
def retrieve(user_idx, n_candidates=20):
    tt_model.eval()
    with torch.no_grad():

        user_scores = tt_model(torch.tensor([user_idx]))[0].clone()

    for idx in seen_by_user[user_idx]:
        user_scores[idx] = -1e9

    candidate_idxs = torch.topk(user_scores, k=n_candidates).indices.tolist()
    return candidate_idxs

In [35]:
def recommend_pipeline(user_idx, n_candidates=20, top_k=5):

    candidates = retrieve(user_idx, n_candidates=n_candidates)
    ranked_candidates = rank_ncf(user_idx, candidates)

    final_top_k = ranked_candidates[:top_k]
    return candidates, final_top_k


In [36]:
demo_users = list(val_pos.keys())[:2]

for u_idx in demo_users:
    real_uid = users[u_idx]
    print(f"РЕКОМЕНДАЦІЙ ДЛЯ КОРИСТУВАЧА №{real_uid}")

    candidates, final_recommendations = recommend_pipeline(user_idx=u_idx, n_candidates=15, top_k=5)

    print(f"Етап 1: Retrieval (Two-Tower відібрала 15 кандидатів):")
    for i, c_idx in enumerate(candidates, start=1):
        title = title_of.get(items[c_idx], "Невідома книга")
        print(f"  {i:02d}. {title}")

    print(f"Етап 2: Ranking (NCF відсортувала та обрала ТОП-5 найкращих):")
    for rank, (c_idx, prob) in enumerate(final_recommendations, start=1):
        title = title_of.get(items[c_idx], "Невідома книга")
        print(f"  {rank}. {title} | (NCF Score: {prob:.4f})")


РЕКОМЕНДАЦІЙ ДЛЯ КОРИСТУВАЧА №8167
Етап 1: Retrieval (Two-Tower відібрала 15 кандидатів):
  01. The Good Earth (House of Earth, #1)
  02. In a Sunburned Country
  03. Another Bullshit Night in Suck City
  04. Killing Yourself to Live: 85% of a True Story
  05. Sex, Drugs, and Cocoa Puffs: A Low Culture Manifesto
  06. No Logo
  07. Fast Food Nation: The Dark Side of the All-American Meal
  08. Freakonomics: A Rogue Economist Explores the Hidden Side of Everything (Freakonomics, #1)
  09. A Million Little Pieces
  10. The Lost Continent: Travels in Small Town America
  11. Notes from a Small Island
  12. Neither Here nor There: Travels in Europe
  13. The War of Art: Break Through the Blocks & Win Your Inner Creative Battles
  14. Giada's Family Dinners
  15. What to Expect the First Year (What to Expect)
Етап 2: Ranking (NCF відсортувала та обрала ТОП-5 найкращих):
  1. In a Sunburned Country | (NCF Score: 0.8606)
  2. Another Bullshit Night in Suck City | (NCF Score: 0.8606)
  3. Kill

**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

Ділення на 2 етапи необхідно для пришвидення відпрацювання та кращої якості.
Як було вказано Two-Tower швидко відбирає кандидатів, тобто він обмежує вибірку на якій після цього бу вже відбуватись ранжування NCF. Якщо залишити тільки ранжування NCF то процес буде значно довший

---
## Завдання 5. Теоретичний блок (письмові відповіді)

Спираючись на лекцію та на те, що Ви щойно побачили на реальних даних, дайте розгорнуті відповіді в markdown-клітинці нижче.

1. **Чому Recall@10 такий низький?** На реальних даних усі моделі цього ДЗ дають скромний Recall@10. Назвіть щонайменше дві причини (підказки: бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач *міг би* вподобати).
2. **Як покращити якість, не змінюючи архітектуру?** Які додаткові ознаки книг і користувачів з Goodbooks можна було б під'єднати? (автор, рік, середній рейтинг, повний набір тегів через TF-IDF, текстові ембединги опису через BERT...)
3. **Diversity.** Якщо користувач любить фентезі, чому не варто показувати йому 10 фентезі-книг підряд? Як технічно підмішати різноманітність?
4. **Freshness / cold start.** Нова книга має 0 оцінок. Який підхід цього ДЗ зможе рекомендувати її одразу, а який — ні? Чому?
5. **Watch time > CTR (з лекції).** Поясніть, чому YouTube оптимізує час перегляду, а не CTR, і як це технічно вшито у weighted logistic regression.


1. Показник низький через тільки 1 параметр ознак - 12 жанрів (в рамках 1 узагальненого жанру книги не рівноцінні для рекомендацій). Велика розрідженість - користувачі взаємодіють лише з крихітною частиною каталогу книг.

2. Необхідно додати нові ознаки книг. Важливим є автор, адже у середині жанрів великий вплив має певний автор і відповідно може бути рекомендація з урахуванням цього. Також рік, сучасні киги можуть бути більш популярніші ніж ті що вийшли 10-20 років тому.
Текстові ембединги опису через BERT мабуть ні так як описи можуть сильно відрізнятись, а повний набір тегів через TF-IDF може бути гарним варіантом без звужування до тільки 12 жанрів.

3. Показувати тільки один і той же жанр може набриднути користувачу. може додати кілька позицій топ серед усіх книг без урахування жанру, який більше всього рекомендується конкретному користувачу.

4. Всі методи які були вище підійдуть для нової книги якщо в неї буде тег жанру адже всі вони основуються саме на параметрі жанру.

5. YouTube оптимізує час перегляду для того щоб рекомендувати відео які користувач переглядав певний час або частку часу. Якщо враховувати  просто відкриття відео (хоча користувач міг дивитись тільки кілька секунд) то це буде спотворювати рекомендаційний список. Для реалізації цього підходу у якості ваги відео які відкривались отримують час перегляду у секундах .Відповідно відео, яке дивились довго, стає "важливішим" під час тренування.